# Omo Forest NDVI Forecasting Project
Vegetation time-series forecasting using Sentinel-2 NDVI and an LSTM.

**Pipeline:** GEE setup -> NDVI time series extraction -> sequence building -> LSTM training -> evaluation -> visualization.

**Cost:** $0 (Google Earth Engine Community tier + free Colab runtime).

## Step 1: Setup — Earth Engine authentication and AOI

In [ ]:
!pip install geemap --quiet

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Replace 'your-project-id' with your actual GEE Cloud Project ID
ee.Authenticate()
ee.Initialize(project='your-project-id')

print("Earth Engine initialized successfully.")


In [ ]:
# Area of Interest: Omo Forest Reserve, Ogun State, Nigeria
omo_forest = ee.Geometry.Rectangle([4.19, 6.35, 4.65, 7.05])

Map = geemap.Map(center=[6.7, 4.4], zoom=9)
Map.addLayer(omo_forest, {'color': 'red'}, 'Omo Forest AOI (rough)')
Map


## Step 2: Extract monthly NDVI time series

In [ ]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )
    return image.updateMask(mask).divide(10000)


def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)


def get_monthly_ndvi(aoi, start_year=2018, end_year=2026, cloud_threshold=60):
    # cloud_threshold relaxed from 30 -> 60: Omo Forest is tropical rainforest with
    # near-constant cloud cover, so a strict 30% threshold rejected most months
    # outright. 60 keeps more real observations while the per-pixel cloud mask
    # (mask_s2_clouds) still scrubs the actual cloudy pixels out of each image.
    records = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            start = ee.Date.fromYMD(year, month, 1)
            end = start.advance(1, 'month')

            collection = (
                ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterBounds(aoi)
                .filterDate(start, end)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
                .map(mask_s2_clouds)
                .map(add_ndvi)
            )

            count = collection.size().getInfo()
            if count == 0:
                continue

            # Median composite is more robust to residual cloud/shadow noise
            # than mean, especially with a looser cloud threshold.
            monthly_composite = collection.select('NDVI').median()

            median_ndvi = monthly_composite.reduceRegion(
                reducer=ee.Reducer.median(),
                geometry=aoi,
                scale=100,
                maxPixels=1e9
            ).get('NDVI')

            try:
                ndvi_value = median_ndvi.getInfo()
            except Exception:
                ndvi_value = None

            if ndvi_value is not None:
                records.append({
                    'year': year,
                    'month': month,
                    'date': f'{year}-{month:02d}-01',
                    'ndvi': ndvi_value,
                    'image_count': count
                })
                print(f'{year}-{month:02d}: NDVI = {ndvi_value:.4f} ({count} images)')

    return pd.DataFrame(records)


In [ ]:
# Full 2018-2026 range with the relaxed cloud filter. This will take a while
# (roughly 8-9 years x 12 months of GEE calls) but should recover far more
# real observations than the earlier 2023-2026 / 30% threshold run did.
ndvi_df = get_monthly_ndvi(omo_forest, start_year=2018, end_year=2026, cloud_threshold=60)
ndvi_df.head(20)


In [ ]:
ndvi_df.to_csv('omo_ndvi_monthly.csv', index=False)

n_possible_months = (2026 - 2018 + 1) * 12
print(f"Real observed months: {len(ndvi_df)} out of {n_possible_months} possible.")
print(f"Saved {len(ndvi_df)} monthly records to omo_ndvi_monthly.csv")


## Step 3: Explore and clean the time series

In [ ]:
ndvi_df['date'] = pd.to_datetime(ndvi_df['date'])
ndvi_df = ndvi_df.sort_values('date').reset_index(drop=True)

plt.figure(figsize=(12, 4))
plt.plot(ndvi_df['date'], ndvi_df['ndvi'], marker='o')
plt.title('Monthly Mean NDVI — Omo Forest Reserve')
plt.xlabel('Date')
plt.ylabel('NDVI')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(ndvi_df['ndvi'].describe())


In [ ]:
# Fill gaps (months with no clear imagery) via interpolation so the LSTM sequence
# is continuous -- but explicitly flag which points are real vs interpolated,
# since that distinction matters for honestly reporting model performance.
ndvi_df = ndvi_df.set_index('date').asfreq('MS')
ndvi_df['is_observed'] = ndvi_df['ndvi'].notna()
ndvi_df['ndvi'] = ndvi_df['ndvi'].interpolate(method='linear')
ndvi_df = ndvi_df.reset_index()

n_observed = ndvi_df['is_observed'].sum()
n_total = len(ndvi_df)
print(f"Real observed months: {n_observed} / {n_total} ({n_observed / n_total:.0%})")
print(f"Interpolated (filled) months: {n_total - n_observed}")

ndvi_df[['date', 'ndvi', 'is_observed']].head(10)


## Step 4: Build sequences for the LSTM
Use a sliding window of past months to predict the next month's NDVI.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

WINDOW_SIZE = 6  # number of past months used to predict the next one
N_FEATURES = 3   # NDVI (scaled) + month_sin + month_cos

# Seasonality features: encode calendar month as a point on a circle so the
# model can learn recurring seasonal patterns (e.g. dry-season dips) instead
# of only reacting to changes after they've already happened.
ndvi_df['month_sin'] = np.sin(2 * np.pi * ndvi_df['date'].dt.month / 12)
ndvi_df['month_cos'] = np.cos(2 * np.pi * ndvi_df['date'].dt.month / 12)

# --- FIX (scaler leakage): fit ONLY on the training period ---
# Old code called scaler.fit_transform() on the FULL series before the
# chronological split, letting test-set min/max leak into training.
# Now: chronological split point is decided FIRST on the raw time axis,
# the scaler is fit on the train portion only, then applied to everything.
n_raw = len(ndvi_df)
train_raw_end = int(n_raw * 0.8)
print(f"Raw months: {n_raw} | scaler fit on first {train_raw_end} (train period only)")

ndvi_values = ndvi_df['ndvi'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(ndvi_values[:train_raw_end])  # <-- train-only fit, no test info
scaled_ndvi = scaler.transform(ndvi_values)  # transform all with train-fitted scaler
print(f"Scaler train min/max: {scaler.data_min_[0]:.4f} / {scaler.data_max_[0]:.4f}")

# month_sin/cos are already bounded in [-1, 1], no scaling needed
seasonal_features = ndvi_df[['month_sin', 'month_cos']].values

feature_matrix = np.hstack([scaled_ndvi, seasonal_features])  # shape (n, 3)


def make_sequences(data, window_size, target_col=0):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size, :])
        y.append(data[i + window_size, target_col])
    return np.array(X), np.array(y)


X, y = make_sequences(feature_matrix, WINDOW_SIZE)
# X shape is already (samples, timesteps, features) -- no reshape needed now

print(f"X shape: {X.shape}, y shape: {y.shape}")


In [ ]:
# Chronological train/test split (do NOT shuffle time series data)
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")


## Step 5: Build and train the LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    LSTM(32, activation='tanh', input_shape=(WINDOW_SIZE, N_FEATURES), return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=8,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training History')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Step 6: Evaluate and visualize the forecast

In [ ]:
y_pred = model.predict(X_test)

# Inverse-transform back to real NDVI scale (scaler was fit on train only -- see Step 4)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_actual = scaler.inverse_transform(y_pred)

from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))

print(f"LSTM  -- Test MAE: {mae:.4f} | Test RMSE: {rmse:.4f}")

# --- Baselines: a forecast is only useful if it beats the trivial option ---
# Persistence ("next month = this month") and seasonal-naive ("next month =
# same month last year") are computed on REAL (unscaled) NDVI so they are
# directly comparable to the LSTM metrics above.
full_actual = ndvi_df['ndvi'].values  # real NDVI, chronological
n_test = len(y_test_actual)
# y_test targets correspond to the last n_test months of the series
test_actual_1d = y_test_actual.flatten()

# Persistence: predict previous month's value (first test point uses last train value)
history_for_persist = full_actual[-(n_test + 1):-1]
persist_pred = history_for_persist.reshape(-1, 1)
mae_persist = mean_absolute_error(y_test_actual, persist_pred)
rmse_persist = np.sqrt(mean_squared_error(y_test_actual, persist_pred))
print(f"Persistence (lag-1)      -- Test MAE: {mae_persist:.4f} | Test RMSE: {rmse_persist:.4f}")

# Seasonal-naive: predict the value from 12 months ago
if n_test + 12 <= len(full_actual):
    seasonal_pred = full_actual[-(n_test + 12):-12].reshape(-1, 1)
    mae_seas = mean_absolute_error(y_test_actual, seasonal_pred)
    rmse_seas = np.sqrt(mean_squared_error(y_test_actual, seasonal_pred))
    print(f"Seasonal-naive (lag-12)  -- Test MAE: {mae_seas:.4f} | Test RMSE: {rmse_seas:.4f}")
else:
    print("Seasonal-naive skipped: not enough history for a 12-month lag.")

# How much of the test set is real observation vs interpolated filler --
# this context matters when judging how meaningful the error metrics are.
test_observed_flags = ndvi_df['is_observed'].values[-len(y_test):]
print(f"Test set real observations: {test_observed_flags.sum()} / {len(test_observed_flags)}")


In [ ]:
test_dates = ndvi_df['date'].values[-len(y_test):]

plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test_actual, label='Actual NDVI', marker='o')
plt.plot(test_dates, y_pred_actual, label='Predicted NDVI', marker='x')
plt.title('LSTM NDVI Forecast vs Actual — Omo Forest Reserve')
plt.xlabel('Date')
plt.ylabel('NDVI')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Step 6b: Rolling-origin cross-validation (robust error estimate)

The single 80/20 split above leaves only ~19 test points, so one lucky or
unlucky year dominates the reported MAE/RMSE. Here we use **expanding-window
rolling-origin CV**: 4 folds, each fold trains on all data up to an origin
and tests on the next 10 months, then the origin rolls forward.

Leakage controls per fold:
- The `MinMaxScaler` is **refit on the fold-train raw period only**.
- Early stopping uses a held-out slice of the **fold-train** (`validation_split`),
  never the fold-test window (the single-split training in Step 5 uses the test
  set for early stopping, which is mildly optimistic — this CV does not).
- A **fresh model** is built per fold so weights never carry across origins.

We report per-fold LSTM vs persistence (lag-1) vs seasonal-naive (lag-12),
plus mean ± std across folds. If the LSTM's mean error is not clearly below
the baselines, the honest conclusion is that it does not beat them yet.


In [ ]:
# --- Step 6b: Rolling-origin (expanding window) cross-validation ---
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import tensorflow as tf

N_SPLITS_CV = 4
TEST_WINDOW = 10   # months per fold test window (4 folds x 10 = 40 test points total)
MIN_TRAIN_SEQ = 55 # sequences in the smallest fold-train (leaves room for 4x10 windows out of 95)
EPOCHS_CV = 100
BATCH_CV = 8

tf.keras.utils.set_random_seed(42)


def build_lstm(window_size=WINDOW_SIZE, n_features=N_FEATURES):
    m = Sequential([
        LSTM(32, activation='tanh', input_shape=(window_size, n_features),
             return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1),
    ])
    m.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return m


raw_ndvi = ndvi_df['ndvi'].values.astype(float)          # real NDVI, chronological
seas = ndvi_df[['month_sin', 'month_cos']].values        # deterministic calendar features
n_raw = len(raw_ndvi)

# Sequence count must satisfy: MIN_TRAIN_SEQ + N_SPLITS_CV * TEST_WINDOW <= n_seq
n_seq = n_raw - WINDOW_SIZE
assert MIN_TRAIN_SEQ + N_SPLITS_CV * TEST_WINDOW <= n_seq, (
    f"Not enough sequences ({n_seq}) for {N_SPLITS_CV}x{TEST_WINDOW} folds "
    f"with min-train {MIN_TRAIN_SEQ}. Reduce folds or window."
)

results = []
for fold in range(N_SPLITS_CV):
    train_end_seq = MIN_TRAIN_SEQ + fold * TEST_WINDOW      # end of fold-train (sequence idx)
    test_end_seq = train_end_seq + TEST_WINDOW

    # --- Per-fold scaler: fit on fold-train RAW period only ---
    # Sequence train_end_seq's last target uses raw row train_end_seq + WINDOW_SIZE - 1,
    # so the raw cutoff below covers exactly what fold-train sees -- nothing after.
    fold_train_raw_end = train_end_seq + WINDOW_SIZE
    sc = MinMaxScaler()
    sc.fit(raw_ndvi[:fold_train_raw_end].reshape(-1, 1))
    scaled_full = sc.transform(raw_ndvi.reshape(-1, 1))
    feat = np.hstack([scaled_full, seas])

    X_all = np.array([feat[i:i + WINDOW_SIZE] for i in range(n_seq)])
    y_all = np.array([feat[i + WINDOW_SIZE, 0] for i in range(n_seq)])
    Xtr, ytr = X_all[:train_end_seq], y_all[:train_end_seq]
    Xte, yte = X_all[train_end_seq:test_end_seq], y_all[train_end_seq:test_end_seq]

    K.clear_session()
    m = build_lstm()
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    hist = m.fit(Xtr, ytr, validation_split=0.15, epochs=EPOCHS_CV,
                 batch_size=BATCH_CV, callbacks=[es], verbose=0)
    epochs_run = len(hist.history['loss'])

    # --- Scores on REAL NDVI scale ---
    yte_actual = sc.inverse_transform(yte.reshape(-1, 1))
    pred_actual = sc.inverse_transform(m.predict(Xte, verbose=0))
    mae_lstm = mean_absolute_error(yte_actual, pred_actual)
    rmse_lstm = float(np.sqrt(mean_squared_error(yte_actual, pred_actual)))

    # Baselines use real NDVI around the fold window (no scaling involved).
    # Targets yte correspond to raw rows [train_end_seq + W .. test_end_seq + W).
    raw_test = raw_ndvi[train_end_seq + WINDOW_SIZE:test_end_seq + WINDOW_SIZE]
    persist = raw_ndvi[train_end_seq + WINDOW_SIZE - 1:test_end_seq + WINDOW_SIZE - 1]
    mae_p = mean_absolute_error(raw_test, persist)
    rmse_p = float(np.sqrt(mean_squared_error(raw_test, persist)))

    if train_end_seq + WINDOW_SIZE - 12 >= 0:
        seas_base = raw_ndvi[train_end_seq + WINDOW_SIZE - 12:test_end_seq + WINDOW_SIZE - 12]
        mae_s = mean_absolute_error(raw_test, seas_base)
        rmse_s = float(np.sqrt(mean_squared_error(raw_test, seas_base)))
    else:
        mae_s, rmse_s = float('nan'), float('nan')

    # Calendar span of this fold's test window, for the report
    d0 = ndvi_df['date'].iloc[train_end_seq + WINDOW_SIZE]
    d1 = ndvi_df['date'].iloc[test_end_seq + WINDOW_SIZE - 1]
    results.append(dict(fold=fold + 1, test_from=str(d0.date()), test_to=str(d1.date()),
                        epochs=epochs_run, mae_lstm=mae_lstm, rmse_lstm=rmse_lstm,
                        mae_p=mae_p, rmse_p=rmse_p, mae_s=mae_s, rmse_s=rmse_s))
    print(f"Fold {fold + 1} [{d0.date()} -> {d1.date()}] "
          f"epochs={epochs_run:3d} | LSTM MAE={mae_lstm:.4f} RMSE={rmse_lstm:.4f} | "
          f"persist MAE={mae_p:.4f} RMSE={rmse_p:.4f} | seas-naive MAE={mae_s:.4f} RMSE={rmse_s:.4f}")

res = pd.DataFrame(results)
print("\nMean +- std across folds:")
for col in ['mae_lstm', 'rmse_lstm', 'mae_p', 'rmse_p', 'mae_s', 'rmse_s']:
    print(f"  {col:9s}: {res[col].mean():.4f} +- {res[col].std():.4f}  (n={len(res)})")

# Compact visual: mean RMSE with std bars
fig, ax = plt.subplots(figsize=(8, 4))
labels = ['LSTM', 'Persist\n(lag-1)', 'Seas-naive\n(lag-12)']
means = [res['rmse_lstm'].mean(), res['rmse_p'].mean(), res['rmse_s'].mean()]
stds = [res['rmse_lstm'].std(), res['rmse_p'].std(), res['rmse_s'].std()]
ax.bar(labels, means, yerr=stds, capsize=6, color=['seagreen', 'gray', 'darkorange'], alpha=0.85)
ax.set_title('Rolling-origin CV: mean RMSE +- std (4 folds x 10 months)', fontweight='bold')
ax.set_ylabel('RMSE (NDVI)')
plt.tight_layout()
plt.savefig('chart_rolling_cv.png', dpi=200)
plt.show()
print("Saved chart_rolling_cv.png")


## Step 7 (Optional): Forecast forward beyond available data

In [ ]:
def forecast_future(model, last_window, scaler, last_date, n_steps=6):
    predictions = []
    current_window = last_window.copy()  # shape (WINDOW_SIZE, N_FEATURES)

    for step in range(1, n_steps + 1):
        pred = model.predict(current_window.reshape(1, WINDOW_SIZE, N_FEATURES), verbose=0)
        pred_ndvi_scaled = pred[0, 0]
        predictions.append(pred_ndvi_scaled)

        # Compute the real calendar month for this forecast step, so the
        # seasonality features stay meaningful instead of being reused/stale.
        future_date = last_date + pd.DateOffset(months=step)
        month_sin = np.sin(2 * np.pi * future_date.month / 12)
        month_cos = np.cos(2 * np.pi * future_date.month / 12)

        new_row = np.array([pred_ndvi_scaled, month_sin, month_cos])
        current_window = np.vstack([current_window[1:], new_row])

    predictions = np.array(predictions).reshape(-1, 1)
    return scaler.inverse_transform(predictions)


last_window = feature_matrix[-WINDOW_SIZE:, :]
last_date = ndvi_df['date'].iloc[-1]
future_ndvi = forecast_future(model, last_window, scaler, last_date, n_steps=6)

print("Forecasted NDVI for next 6 months:")
for i, val in enumerate(future_ndvi.flatten(), start=1):
    print(f"  Month +{i}: {val:.4f}")


## Step 8: Spatial NDVI Visualization
The time series shows *when* vegetation health changed, but not *where*. This step renders the actual NDVI raster as a color-coded map so degraded vs healthy zones within Omo Forest are visible directly.

In [ ]:
def get_ndvi_composite_for_month(aoi, year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
        .map(mask_s2_clouds)
        .map(add_ndvi)
    )
    return collection.select('NDVI').median()


# Pick two months to compare -- e.g. a healthy-looking month vs a degraded one,
# based on the values already printed in Step 2's output.
ndvi_healthy = get_ndvi_composite_for_month(omo_forest, 2024, 5)   # example: high NDVI month
ndvi_degraded = get_ndvi_composite_for_month(omo_forest, 2025, 1)  # example: low NDVI month

vis_params = {'min': 0.2, 'max': 0.8, 'palette': ['brown', 'yellow', 'green']}

Map = geemap.Map(center=[6.7, 4.4], zoom=10)
Map.addLayer(ndvi_healthy, vis_params, 'NDVI - Healthier Month (2024-05)')
Map.addLayer(ndvi_degraded, vis_params, 'NDVI - Degraded Month (2025-01)')
Map.add_colorbar(vis_params, label='NDVI')
Map


In [ ]:
# Export a static snapshot for the writeup/README (no interactive controls needed there)
geemap.ee_export_image(
    ndvi_healthy.clip(omo_forest),
    filename='ndvi_healthy_month.tif',
    scale=100,
    region=omo_forest
)
print("Exported ndvi_healthy_month.tif -- open in QGIS or convert to PNG for the README.")


## Optional Enrichment: FAOSTAT Nigeria yield correlation
If you want to relate NDVI trends to real agricultural yield, download free Nigeria crop yield data from [FAOSTAT](https://www.fao.org/faostat/en/#data/QCL) and merge by year for a coarse correlation check. This is a stretch goal, not required for the core forecasting deliverable.

## Step 9: Export polished charts for README / logbook
Publication-style versions of the key charts, saved as PNG files ready to embed in a GitHub README or paste into the logbook writeup.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

# --- Chart 1: Full NDVI history, observed vs interpolated ---
fig, ax = plt.subplots(figsize=(12, 5))
observed = ndvi_df[ndvi_df['is_observed']]
interpolated = ndvi_df[~ndvi_df['is_observed']]

ax.plot(ndvi_df['date'], ndvi_df['ndvi'], color='seagreen', linewidth=1.5, zorder=1)
ax.scatter(observed['date'], observed['ndvi'], color='seagreen', s=35, label='Observed', zorder=2)
ax.scatter(interpolated['date'], interpolated['ndvi'], color='lightgray', s=35,
           label='Interpolated', zorder=2, edgecolors='gray')

ax.set_title('Omo Forest Reserve — Monthly NDVI (2018-2026)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('NDVI')
ax.legend()
plt.tight_layout()
plt.savefig('chart_ndvi_history.png', dpi=200)
plt.show()
print("Saved chart_ndvi_history.png")


In [ ]:
# --- Chart 2: Forecast vs Actual, polished ---
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(test_dates, y_test_actual, color='seagreen', marker='o', linewidth=2,
        label='Actual NDVI', markersize=7)
ax.plot(test_dates, y_pred_actual, color='darkorange', marker='x', linewidth=2,
        linestyle='--', label='LSTM Forecast', markersize=8)

ax.set_title('LSTM NDVI Forecast vs Actual — Omo Forest Reserve', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('NDVI')
ax.legend()
ax.text(
    0.02, 0.02, f'MAE: {mae:.3f}  |  RMSE: {rmse:.3f}',
    transform=ax.transAxes, fontsize=10,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
)
plt.tight_layout()
plt.savefig('chart_forecast_vs_actual.png', dpi=200)
plt.show()
print("Saved chart_forecast_vs_actual.png")


In [ ]:
# --- Chart 3: Training history, polished ---
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(history.history['loss'], color='steelblue', linewidth=2, label='Train Loss')
ax.plot(history.history['val_loss'], color='darkorange', linewidth=2, label='Val Loss')
ax.set_title('LSTM Training History', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.savefig('chart_training_history.png', dpi=200)
plt.show()
print("Saved chart_training_history.png")
print("\nAll charts saved. Download them from the Colab file browser (folder icon, left sidebar).")
